In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base_model_name = "models/gemma-2b-it"
adapter_path = "models/gemma-2b-it-fine-tuned-edit"

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.float16,
    device_map={"": "cuda:0"}    # FIX: force entire model onto single GPU
)

tokenizer = AutoTokenizer.from_pretrained(base_model_name)

model = PeftModel.from_pretrained(
    base_model, 
    adapter_path,
    device_map={"": "cuda:0"}    # FIX: adapter on same device
)

model = model.merge_and_unload()
model.eval()
print("Model loaded on:", next(model.parameters()).device)

/home/teaching/miniconda3/envs/dl45/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.11it/s]


Model loaded on: cuda:0


In [2]:
import json

def edit_prompt(current_json, instruction):
    return f"""
### SYSTEM:
You are an AI system designed to MODIFY an existing job description JSON.

Your task:
- Update the given JSON based ONLY on the user instruction.
- Do NOT regenerate the entire job description.
- Make ONLY the necessary changes.

---

RULES:
- Output MUST be valid JSON only.
- Return ONLY ONE JSON object.
- Do NOT include explanations or extra text.
- Do NOT change fields that are not related to the instruction.
- Preserve all existing data unless modification is required.
- Do NOT hallucinate new fields or unnecessary content.

---

EDITING GUIDELINES:

1. ADD:
- Add new items to the correct field without removing existing ones.

2. REMOVE:
- Remove only the specified content.
- Do NOT delete unrelated items.

3. UPDATE:
- Modify only the specified field value.

4. REPLACE:
- Replace only the mentioned parts.

5. REFINE:
- Improve wording while preserving meaning.

6. IMPROVE:
- Make content more professional or detailed without changing intent.

7. REGENERATE:
- Rewrite the entire JSON ONLY if explicitly requested.

---

### USER:

Existing JSON:
{json.dumps(current_json, indent=2)}

Instruction:
{instruction}

---

### RESPONSE:
"""

def extract_first_json(output: str):
    start = output.find("{")
    if start == -1:
        return None

    brace_count = 0
    for i in range(start, len(output)):
        if output[i] == "{":
            brace_count += 1
        elif output[i] == "}":
            brace_count -= 1
        if brace_count == 0:
            return output[start:i+1]
    return None

def edit_jd(state, instruction):
    prompt = edit_prompt(state, instruction)
    
    # FIX: explicitly move to cuda
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to("cuda") for k, v in inputs.items()}

    outputs = model.generate(
        **inputs,
        max_new_tokens=600,
        do_sample=False,
        repetition_penalty=1.2
    )

    prompt_len = inputs["input_ids"].shape[1]
    new_tokens = outputs[0][prompt_len:]
    text = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return extract_first_json(text)

In [ ]:


state = {'job_title': 'Python Developer',
'location': 'N/A',
'industry': '',
'responsibilities': ['Design, develop, and maintain robust Python applications for our clients.', 'Collaborate closely with cross-functional teams to define project requirements and timelines.', 'Write clean, efficient, and well-documented code that adheres to best practices.', 'Troubleshoot and resolve technical issues related to application performance and functionality.', 'Participate in code reviews and provide constructive feedback to peers.', 'Ensure high quality deliverables through rigorous testing and validation procedures.'],
'requirements': ['Minimum of 2 years of experience as a Python developer.', 'Proficiency in Object Oriented Programming principles and data structures.', 'Strong understanding of web services and RESTful APIs.', 'Experience with popular frameworks such as Django or Flask is preferred.', 'Excellent communication skills and ability to collaborate effectively with team members.'],
'qualifications': '',
'experience': ['At least 2 years of experience developing software solutions using Python.', 'Familiarity with cloud computing platforms like AWS or Azure is desirable.'],
'other_requirements': ['Ability to adapt to changing priorities and deadlines.', 'Commitment to continuous learning and staying updated with emerging technologies.']}

while True:
    user_input = input("\n>> ").strip()

    if user_input == "exit":
        break

    result = edit_jd(state, user_input)

    if result is None:
        print("Model returned invalid JSON, try again")
        continue

    # FIX: parse result back to dict so next iteration sends proper JSON
    try:
        state = json.loads(result)
    except json.JSONDecodeError:
        print("Could not parse model output, state unchanged")
        print("Raw output:", result[:300])
        continue

    print("\nUPDATED JD:\n", json.dumps(state, indent=2))

The 'batch_size' attribute of HybridCache is deprecated and will be removed in v4.49. Use the more precisely named 'self.max_batch_size' attribute instead.



UPDATED JD:
 {
  "job_title": "Python Developer",
  "location": "Mandi",
  "industry": "",
  "responsibilities": [
    "Design, develop, and maintain robust Python applications for our clients.",
    "Collaborate closely with cross-functional teams to define project requirements and timelines.",
    "Write clean, efficient, and well-documented code that adheres to best practices.",
    "Troubleshoot and resolve technical issues related to application performance and functionality.",
    "Participate in code reviews and provide constructive feedback to peers.",
    "Ensure high quality deliverables through rigorous testing and validation procedures."
  ],
  "requirements": [
    "Minimum of 2 years of experience as a Python developer.",
    "Proficiency in Object Oriented Programming principles and data structures.",
    "Strong understanding of web services and RESTful APIs.",
    "Experience with popular frameworks such as Django or Flask is preferred.",
    "Excellent communication 

: 